# Rete Pix2Pix
In questo notebook analizziamo il funzionamento dello script `Pix2Pix.py`, responsabile dell'addestramento e della validazione di una rete neurale generativa per la colorazione virtuale di immagini istopatologiche. Il codice implementa un'architettura Pix2Pix con generatore **U-Net** e discriminatore **PatchGAN**, addestrata su coppie di immagini allineate label-free \- stained.

## Passaggi preliminari

### Pacchetti necessari
I pacchetti necessari per lo script `Pix2Pix.py` sono:

- `numpy`
- `pytorch`
- `torchvision`

In [ ]:
# Pacchetti standard
import os, random, time, datetime, sys
from PIL import Image

# Pacchetti esterni
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
from torchvision.utils import save_image

## Dataset
Per addestrare correttamente una rete Pix2Pix, è fondamentale disporre di dati organizzati in coppie di immagini perfettamente allineate in cui ad ogni immagine label-free (non colorata) corrisponde la sua controparte stained (colorata).
Nel nostro caso, i dati istopatologici sono organizzati in file `.tif`, dove ogni coppia condivide un prefisso comune nel nome (es. 00240_00100_label_free.tif e 00240_00100_stained.tif). Questo lavoro è eseguito dalla funzione `_get_pairs()`
La creazione del dataset, in tutti i suoi step, è illustrata in `ollie_wan_kenobi.ipynb`.

In questa sezione definiamo una classe `PairedHistologyDataset`, ereditaria della classe `Dataset` di **pytorch**, che identifica e carica automaticamente tutte le coppie di immagini presenti nella cartella fornita.

Questa struttura consente di gestire dataset di immagini istologiche accoppiate in modo efficiente e modulare, rendendole pronte per il training supervisionato della rete neurale.

In [ ]:
class PairedHistologyDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.pairs = self._get_pairs()

    def _get_pairs(self):
        files = os.listdir(self.folder_path)
        prefixes = [f.replace('_label_free.tif', '')
                    for f in files if f.endswith('_label_free.tif')
                    and f.replace('_label_free.tif', '') + '_stained.tif' in files]
        return sorted(prefixes)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prefix = self.pairs[idx]
        lf = Image.open(os.path.join(self.folder_path, prefix + '_label_free.tif')).convert('RGB')
        st = Image.open(os.path.join(self.folder_path, prefix + '_stained.tif')).convert('RGB')
        if self.transform:
            lf = self.transform(lf)
            st = self.transform(st)
        return lf, st

Le funzioni `__len__(self)` e `__getitem__(self, idx)` sono implementate per rispettare le specifiche della classe `Dataset` di **pytorch**. La funzione `__len__(self)` restituisce il numero di coppie di immagini nel dataset, mentre `__getitem__(self, idx)` carica e restituisce la coppia di immagini corrispondente all'indice `idx`.

## Conf Setting JSON (DA FARE)

## Generatore
Il generatore è il componente della rete che si occupa di tradurre un'immagine non colorata (label-free) in una versione virtualmente colorata, simulando l'effetto di una colorazione istopatologica H&E.
Per farlo, utilizziamo un'architettura chiamata U-Net, particolarmente efficace nelle trasformazioni _image to image_ perché riesce a preservare sia le strutture locali che la coerenza globale.

In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self, n_channels=3, n_classes=3, bilinear=False):
        super(UNetGenerator, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        # Encoders
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        # Decoders
        self.up1 = Up(1024, 512, bilinear)
        self.up2 = Up(512, 256, bilinear)
        self.up3 = Up(256, 128, bilinear)
        self.up4 = Up(128, 64, bilinear)

        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        logits = self.outc(x)
        return logits


La U-Net è composta da due parti principali:


- Encoder: comprime progressivamente l’immagine riducendo la sua risoluzione ma aumentando il numero di canali (**profondità**). In pratica, estrae caratteristiche astratte e informazioni ad alto livello.

- Decoder: ricostruisce l’immagine riportandola alla risoluzione originale, tentando di riprodurre una versione colorata realistica dell’input.

Per comprendere a fondo l'implementazione del generatore è necessario analizzare i moduli che compongono la U-Net.

### DoubleConv
Il blocco DoubleConv è l’unità base dell’architettura U-Net. È composto da due operazioni di convoluzione sequenziali, ciascuna strutturata come segue:

In [ ]:
nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
nn.BatchNorm2d(out_channels),
nn.ReLU(inplace=True)


- `Conv2d()` è un layer convoluzionale bidimensionale che applica un _kernel_ (una matrice di pesi che viene utilizzata per scansionare un'immagine al fine di calcolare nuove rappresentazioni locali) all'immagine per estrarne caratteristiche locali. I suoi argomenti sono:
    
    - `in_channels`: indica il numero di canali dell'immagine in ingresso (es. 3 per RGB, 64 dopo la prima convoluzione, _si guardi il primo encoder della classe UNetGenerator_)

    - `out_channels`: indica il numero di filtri da applicare, cioé il numero di feature map in uscita. Determina la **profondità** dell'output

    - `kernel_size`: indica la dimensione del filtro, di default impostiamo questo valore a 3 per opprenere un filtro 3x3 da applicare all'immagine

    - `stride`: indica il passo di scorrimento del kernel sull’immagine. Stride maggiori riducono le dimensioni spaziali dell’output. Di default è impostato a 1 permettendoci di conservare la massima densità di informazione
    
    - `padding`: aggiunge un bordo di 1 pixel per lato. Serve per mantenere le dimensioni spaziali invariate copo la convoluzione: $output\_ size=\lfloor \frac{input\_ size-kernel\_ size+2\cdot padding}{stride}+1\rfloor$

    - `bias`: disabilita il termine di bias. È comune se si usa `BatchNorm2d` subito dopo, perché quest’ultima ha già un termine di shift e scale che sostituisce il ruolo del bias.

- `BatchNorm2d()`: È una normalizzazione per batch sui canali, che accelera e stabilizza l'addestramento. I suoi parametri sono:

    - `out_channels`: indica il numero di canali in uscita dalla convoluzione, che è ciò su cui viene applicata la normalizzazione

- `ReLU()`: abbreviazione di Rectified Linear Unit, è una funzione di attivazione non lineare. La sua presenza è necessaria perché, senza una funzione di attivazione, anche più strati convoluzionali si ridurrebbero a una singola trasformazione lineare, limitando drasticamente la capacità della rete di apprendere relazioni complesse. La funzione agisce introducendo non linearità nel modello, annullando tutti i valori negativi generati dai layer precedenti: $ReLU(x)=max(0, x)$. In termini interpretativi, i valori negativi possono essere visti come assenza o opposizione rispetto a certe caratteristiche rilevate dal kernel, motivo per cui vengono eliminati. Tra i suoi parametri troviamo:
    
    - `inplace`: il quale permette di sovrascrivere i valori di input per risparmiare memoria. Non influisce sul comportamento numerico

Il blocco `DoubleConv` utilizza due convoluzioni consecutive per migliorare la capacità espressiva del modello. La prima convoluzione serve a trasformare l'immagine in uno spazio di caratteristiche più profondo (in_channels → out_channels), mentre la seconda mantiene questo spazio (out_channels → out_channels) e lo raffina attraverso ulteriori trasformazioni non lineari. Questo schema consente di apprendere pattern più ricchi ed evitare che l’informazione venga compressa prematuramente.  

### Down
rappresenta un passaggio di compressione all’interno dell’encoder. Ogni blocco applica prima un’operazione di max pooling con kernel 2×2 (che dimezza le dimensioni spaziali) seguita da un blocco DoubleConv

### Up


### OutConv

Ogni passaggio è realizzato tramite blocchi convoluzionali (filtri che estraggono pattern) e funzioni di attivazione (come ReLU), che trasformano i dati mantenendone la struttura.



Durante la compressione, però, si rischia di perdere dettagli importanti (es. bordi cellulari o nuclei).
Per questo, la U-Net utilizza delle connessioni di salto: collegamenti diretti tra ogni livello dell’encoder e il corrispondente livello del decoder.

Queste connessioni servono per riutilizzare i dettagli catturati all'inizio e "fonderli" nella fase di ricostruzione, migliorando la qualità dell'immagine generata.

Tecnicamente, ciò avviene concatenando le feature map in profondità lungo l’asse dei canali.

Durante la compressione, però, si rischia di perdere dettagli importanti (es. bordi cellulari o nuclei).
Per questo, la U-Net utilizza delle connessioni di salto: collegamenti diretti tra ogni livello dell’encoder e il corrispondente livello del decoder.

Queste connessioni servono per riutilizzare i dettagli catturati all'inizio e "fonderli" nella fase di ricostruzione, migliorando la qualità dell'immagine generata.

Tecnicamente, ciò avviene concatenando le feature map in profondità lungo l’asse dei canali.

## Discriminatore

## Funzioni di perdita

## Addestramento

### Validazione

### Checkpoint

## Test

### Metriche (loss)

## Risultati